# UKHousing: SVGP, GPW, standard Patchwork and VNNGP

Upload to Colab, select a GPU runtime and Run all. Restart the runtime
first if an older copy has already loaded model classes.
Code loads from the latest main of chahineNejm/Generalized-Patchwork.
The resolved commit and source archive are saved with the results.

This reconstructs the 2018 flats task from HM Land Registry transactions
and FreeMapTools postcode centroids using the HIP-GP authors' region and
filters. The verified reconstruction contains 185,254 transactions.
It is NOT the verified original VNNGP dataset snapshot. Do not compare
these numbers directly with the VNNGP paper's table.
Downloads are checksum-pinned: changed files raise errors, not substitutions.

All four models use the same seeded 64/16/20 train/validation/test split,
Gaussian observation likelihood, ARD RBF kernel family and float64.
Only training targets determine log-price standardization. Coordinates
use the authors' fixed geographic ROI. Repeated-coordinate transactions
are retained, so random-split performance is not spatial extrapolation.
VNNGP's paper instead uses Matern-5/2: this is a controlled RBF comparison.

Default runs: SVGP 1,024 inducing points; GPW 8 x 128; standard Patchwork
conditions directly on all training targets in fixed rectangular regions
with at most 512 observations each; VNNGP has one inducing point per
training transaction and 32 preceding neighbors. Budgets are displayed,
not represented as equal. Standard Patchwork uses more regions to make
exact local conditioning feasible; this is not a same-partition ablation.

Each model trains for 100 epochs. Variational models freeze kernel/noise
parameters for 20 epochs; exact Patchwork has no variational warm-up and
performs 100 full joint-likelihood updates. No early stopping or
test-based selection. All runs have tqdm and live diagnostics every 10
epochs, then metrics and maps. Final metrics use standardized log prices;
scores in unstandardized log GBP are saved separately. No GBP-scale claims.
Training timings include live diagnostics/checkpoints; prediction timings
cover validation and test inference, excluding plots and scoring.

GPW weights use the local covariance conditional on inducing values and
stitching, independent of q(U). Latent variance includes alpha^T R alpha
and propagated inducing uncertainty. Training uses the Gaussian ELBO.
VNNGP uses GPyTorch 1.15.2 with exact CPU FAISS neighbor searches and GPU
GP calculations. CPU search is an explicit design choice, not a fallback.
Prediction uses the same mean-field conditional equations with two
triangular solves, checked against GPyTorch's original predictions.
The stochastic KL uses the same SPD solver too, replacing the upstream
batched torch.linalg.solve without changing the KL formula or jitter.
Numerical failures raise; no retries, jitter escalation, CPU substitution,
variance clipping or automatic configuration changes are implemented here.


## 1. GitHub source and pinned VNNGP dependencies


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/content/gpw_ukhousing')
if ROOT.exists():
    raise FileExistsError("Checkout already exists; use a fresh Colab runtime")
if any(name == "pwgp" or name.startswith("pwgp.") or name == "experiments"
       or name.startswith("experiments.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading a new source version")
os.environ["GIT_TERMINAL_PROMPT"] = "0"
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, "--single-branch",
                REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import subprocess, sys
_GPW_CELL_RESULT = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "/content/gpw_ukhousing[ukhousing]"], check=True)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 2. Settings and GPU checks


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import gc, json, shutil
from dataclasses import asdict, replace
from datetime import UTC, datetime
import faiss
import gpytorch
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display
from experiments.ukhousing_benchmark import UKConfig, archive_run, prepare_split, run_ukhousing
from pwgp.data.ukhousing import prepare_ukhousing
from pwgp.linalg import cholesky_solve
from pwgp.models.vnngp import VNNGP, strict_gpytorch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, restart, then Run all")
DEVICE = "cuda:0"
torch.set_num_threads(2)
faiss.omp_set_num_threads(2)
print({"GPU": torch.cuda.get_device_name(0), "torch": torch.__version__,
       "CUDA": torch.version.cuda, "gpytorch": gpytorch.__version__,
       "neighbor_search": "exact CPU FAISS", "dtype": "float64"})
for batch, size, columns in [(8, 128, 128), (512, 32, 1), (1, 1024, 4)]:
    matrix = (torch.eye(size, dtype=torch.float64, device=DEVICE) + 0.01).expand(batch, size, size).clone()
    matrix.requires_grad_(True)
    rhs = torch.ones(batch, size, columns, dtype=torch.float64, device=DEVICE)
    answer = cholesky_solve(rhs, torch.linalg.cholesky(matrix))
    torch.testing.assert_close(matrix @ answer, rhs)
    answer.square().mean().backward()
    assert torch.isfinite(matrix.grad).all()
del matrix, rhs, answer
GPU_KL_PROBES = [(4, 8), (32, 512)]
for neighbors, batch_size in GPU_KL_PROBES:
    generator = torch.Generator(device=DEVICE).manual_seed(31)
    probe_x = 0.1 + 0.8 * torch.rand(neighbors + batch_size, 2, device=DEVICE,
                                     dtype=torch.float64, generator=generator)
    probe = VNNGP(probe_x, neighbors=neighbors, batch_size=batch_size)
    for step in range(2):
        with strict_gpytorch():
            probe.train()
            probe.zero_grad()
            prediction = probe(None)
            probe_indices = probe.variational_strategy.current_training_indices.to(DEVICE)
            loss = -gpytorch.mlls.VariationalELBO(probe.likelihood, probe, num_data=len(probe_x))(
                prediction, torch.sin(probe_x[probe_indices, 0]))
            loss.backward()
        assert len(probe_indices) == (neighbors if step == 0 else batch_size)
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters())
    probe_mean, probe_variance = probe.predict(probe_x[:7])
    assert torch.isfinite(probe_mean).all() and (probe_variance > 0).all()
    torch.cuda.synchronize()
    del probe, prediction, loss, probe_mean, probe_variance
gc.collect()
torch.cuda.empty_cache()
print("VNNGP first-K and stochastic KL batches, backward and prediction passed")

from experiments.ukhousing_benchmark import build_model
probe_rng = np.random.default_rng(31)
probe_coordinates = probe_rng.uniform(0.05, 0.95, (80, 2))
probe_split = prepare_split(probe_coordinates * [7.5, 5.5] + [-5.7, 50],
                            np.sin(8 * probe_coordinates[:, 0]))
for name in ["gpw", "svgp", "patchwork"]:
    probe_config = UKConfig(model=name, device=DEVICE, inducing_points=12,
                            patches=3, q_rank=4, stitches_per_edge=2,
                            exact_maximum_points=16)
    probe = build_model(probe_config, probe_split)
    probe_x = probe_split["x"][probe_split["train"][:8]].to(DEVICE)
    probe_y = probe_split["y"][probe_split["train"][:8]].to(DEVICE)
    loss = probe.loss() if name == "patchwork" else -probe.elbo_batch(probe_x, probe_y, len(probe_x))[0]
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters())
    probe_mean, probe_variance = probe.predict(probe_x)
    assert torch.isfinite(probe_mean).all() and (probe_variance > 0).all()
    del probe, loss, probe_mean, probe_variance
gc.collect()
torch.cuda.empty_cache()
print("All four model GPU checks passed")

from experiments.default_hyperparameters import ukhousing_defaults
COMMON = replace(ukhousing_defaults(device=DEVICE))  # Add explicit overrides here.
RUNS = [("SVGP 1024", replace(COMMON, model="svgp")),
        ("GPW 8 x 128", replace(COMMON, model="gpw")),
        ("Standard Patchwork", replace(COMMON, model="patchwork")),
        ("VNNGP K32", replace(COMMON, model="vnngp"))]
RUN_ROOT = ROOT / "results/ukhousing" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
(RUN_ROOT / "source_snapshot.zip").write_bytes(SOURCE_ZIP)
(RUN_ROOT / "source_provenance.json").write_text(json.dumps(SOURCE_PROVENANCE, indent=2))
_GPW_CELL_RESULT = display(pd.DataFrame([{"run": name, **asdict(config)} for name, config in RUNS]))
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 3. Download, validate and prepare one shared dataset


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
raw_x, raw_y, manifest = prepare_ukhousing(ROOT / "data/ukhousing")
split = prepare_split(raw_x, raw_y, seed=COMMON.seed)
np.savez_compressed(RUN_ROOT / "split.npz", train=split["train"],
                    validation=split["validation"], test=split["test"])
preprocessing = {key: split[key] for key in
                 ["target_mean", "target_std", "seed", "input_transform", "target_transform"]}
(RUN_ROOT / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2))
(RUN_ROOT / "data_manifest.json").write_text(json.dumps(manifest, indent=2))
display(manifest)
_GPW_CELL_RESULT = print({name: len(split[name]) for name in ["train", "validation", "test"]})
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 4. Train and report each model


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
from google.colab import files

summaries, completed = [], []
for name, config in RUNS:
    display(Markdown(f"### {name}"))
    live = display(Markdown("Initializing model..."), display_id=True)

    def show_progress(path, epoch, handle=live):
        handle.update(Image(filename=str(path / "progress.png")))

    result = run_ukhousing(config, split,
                           directory=RUN_ROOT / name.lower().replace(" ", "_"),
                           on_snapshot=show_progress)
    metrics = json.loads((result / "metrics.json").read_text())
    summaries.append({"model": name, **metrics})
    completed.append(result)
    display(pd.DataFrame([summaries[-1]]).set_index("model").T)
    display(Image(filename=str(result / "maps.png")))
    comparison = pd.DataFrame(summaries).set_index("model")
    comparison.to_csv(RUN_ROOT / "comparison.csv")
    run_archive = archive_run(result)
    gc.collect()
    torch.cuda.empty_cache()
    print("Saved run:", run_archive)
    print("CUDA allocated/reserved MiB:", torch.cuda.memory_allocated() / 2**20,
          torch.cuda.memory_reserved() / 2**20)
    files.download(str(run_archive))
display(Markdown("## Final comparison: standardized log-price metrics"))
_GPW_CELL_RESULT = display(comparison)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 5. Download all results


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
from google.colab import files

archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
print("Results archive:", archive)
_GPW_CELL_RESULT = files.download(archive)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## Sources and attribution
- [UK flats preparation reference](https://github.com/cunningham-lab/hipgp/blob/master/experiments-hip-gp/uk_housing_data.py)
- [HM Land Registry Price Paid Data](https://www.gov.uk/government/statistical-data-sets/price-paid-data-yearly-file)
- [Postcode coordinate source](https://www.freemaptools.com/download-uk-postcode-lat-lng.htm)
- [Patchwork Kriging paper](https://jmlr.org/papers/v19/17-042.html)
- [VNNGP paper](https://proceedings.mlr.press/v162/wu22h.html)
- [GPyTorch VNNGP implementation](https://docs.gpytorch.ai/en/v1.15.2/examples/04_Variational_and_Approximate_GPs/VNNGP.html)

Contains HM Land Registry data (C) Crown copyright and database right 2021.
Licensed under the Open Government Licence v3.0.
Contains Ordnance Survey data (C) Crown copyright and database right 2021.
Contains Royal Mail data (C) Royal Mail copyright and database right 2021.
Source: Office for National Statistics licensed under OGL v3.0; via FreeMapTools.
